## REDUCED RISK KAISER TRIALS

The goal of this notebook is to validate the overall reduced risk of the Kaiser Northern PC population. The risk to CV, dementia and MCI should be, all independently, equal to about 1/3 of the corresponding risks in the KSPC population. A comparison will need to be made between these results and the results from the notebook using the KSPC population.

In [1]:
import os
microsimDir = "/Users/deligkaris.1/OneDrive - The Ohio State University Wexner Medical Center/MICROSIM/CODE/microsim"
os.chdir(microsimDir)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize

from microsim.person_filter import PersonFilter
from microsim.trials.trial_description import KaiserTrialDescription
from microsim.trials.trial_type import TrialType
from microsim.trials.trial import Trial
from microsim.trials.trial_outcome_assessor_factory import TrialOutcomeAssessorFactory
from microsim.treatment_strategies.treatment_strategies import TreatmentStrategiesType
from microsim.default_treatments.default_treatments import DefaultTreatmentsType
from microsim.treatment_strategies.treatment_strategy_repository import TreatmentStrategyRepository
from microsim.treatment_strategies.bp_treatment_strategies import SprintForSbpRiskThreshold
from microsim.treatment_strategies.statin_treatment_strategies import StatinTreatmentStrategy
from microsim.population_factory import PopulationFactory
from microsim.outcomes.cv_model_repository import CVModelRepository
from microsim.outcomes.outcome import OutcomeType
from microsim.risk_factors.modality import Modality
from microsim.treatment_strategies.wmd_treatment_strategies import Wmd15TreatmentStrategy, Wmd20TreatmentStrategy, Wmd25TreatmentStrategy

pd.set_option('future.no_silent_downcasting', True) #make the calculation here future-proof

In [2]:
%%time
pf = PersonFilter() 
pf.add_filter("person", "hasWmh", lambda x: (x.has_wmh()))
pf.add_filter("person", "noMCI", lambda x: not x.has_mci(inSim=False))

wmhSpecific=True

sampleSize = 100000
duration = 4

ts = TreatmentStrategyRepository()
#ts._repository[TreatmentStrategiesType.WMD15.value] = Wmd15TreatmentStrategy()
ts._repository[TreatmentStrategiesType.WMD20.value] = Wmd20TreatmentStrategy()
#ts._repository[TreatmentStrategiesType.WMD25.value] = Wmd25TreatmentStrategy()

td = KaiserTrialDescription(trialType = TrialType.COMPLETELY_RANDOMIZED,
                            blockFactors=list(),
                            sampleSize = sampleSize, 
                            duration = duration, 
                            treatmentStrategies = ts,
                            nWorkers = 5, 
                            personFilters=pf,
                            wmhSpecific=wmhSpecific) 

tr = Trial(td)

tr.print_covariate_distributions()

tr.run()

toa = TrialOutcomeAssessorFactory.get_trial_outcome_assessor()
tr.analyze(toa)

print(tr)

                          Printing covariate information for people still alive...
                          self=treated, alive people count=  100000               other=control, alive people count=  100000  
                          self=treated, unique alive people count=  83096         other=control, unique alive people count=  83270   
                          self                                                    other
                          -----------------------------------------------------   -----------------------------------------------------
                          min      0.25    med     0.75     max    mean     sd     min     0.25    med     0.75     max    mean     sd
                          -----------------------------------------------------   -----------------------------------------------------
                    age    45.0    64.0    71.0    77.0   113.0    70.9     9.5    45.0    64.0    71.0    77.0   113.0    70.8     9.5
                    sbp  